In [ ]:
# =============================================================================
# DAY 26 — Multi-Agent Reinforcement Learning (MARL) Basics
# =============================================================================
# So far (Days 22–25): ONE agent in one environment (CartPole, Pendulum, LLM).
# Today: SEVERAL agents act at once. Their actions change the world together,
# and each may get its own reward (or share one).
#
# Everyday pictures:
#   Cooperative: two robots carrying a table — both must succeed together.
#   Competitive: chess / soccer opponents.
#   Mixed: self-driving cars — share the road but each wants its own destination.
#
# This notebook: concepts first, then three baselines on a tiny 2-agent Gridworld.
#
#
# ---------------------------------------------------------------------------
# 1) FORMAL SETTING — MARKOV GAMES (also called Stochastic Games)
# ---------------------------------------------------------------------------
#
# A Markov Game with N agents is the multi-agent generalization of an MDP.
#
# Ingredients:
#   S          = shared state space (what the world looks like)
#   A_i        = action set for agent i   (i = 1…N)
#   A          = A_1 × A_2 × … × A_N      joint action
#   P(s'|s, a) = transition given the JOINT action a = (a_1,…,a_N)
#   R_i(s, a)  = reward for agent i (may depend on everyone's actions)
#   γ          = discount
#   (optional) observation o_i = O_i(s) if agents don't see full state
#
# Special cases:
#   N = 1                         → ordinary MDP
#   all R_i equal (team reward)   → fully cooperative
#   R_1 = −R_2 (two players)      → zero-sum competitive
#   otherwise                     → general-sum / mixed
#
# Each agent i has a policy π_i(a_i | o_i) (or | s if fully observed).
# The joint policy is π = (π_1,…,π_N).
#
# Goal (cooperative view we use today):
#   maximize expected TEAM return  E[ Σ_t γ^t r_team(s_t, a_t) ]
#
# Important: from agent i's view, the "environment" includes the OTHER agents.
# If they change their policies while i learns, the world looks NON-STATIONARY.
# That is the root of most MARL pain.
#
#
# ---------------------------------------------------------------------------
# 2) THE CORE CHALLENGES OF MARL
# ---------------------------------------------------------------------------
#
# (A) Non-stationarity
#     Single-agent RL assumes a fixed P and R. In MARL, when teammate/opponent
#     updates π_j, the effective dynamics for i change. Experience goes stale.
#
# (B) Credit assignment (multi-agent)
#     Team got +10 — was it because of my move or yours? Harder than single-agent
#     temporal credit assignment.
#
# (C) Scalability of joint action spaces
#     |A| = |A_1| · |A_2| · … grows exponentially. Joint Q(s, a_1, a_2, …)
#     is only feasible for tiny discrete spaces (our "centralized Q" baseline).
#
# (D) Partial observability
#     Often each agent sees only a local view. Need policies over observations,
#     sometimes with memory (RNNs) — we keep Gridworld fully observed for clarity.
#
# (E) Equilibrium / which solution?
#     In competitive or mixed games there may be many Nash equilibria.
#     Cooperative MARL usually sidesteps this by optimizing a shared reward.
#
#
# ---------------------------------------------------------------------------
# 3) PARADIGMS (how people structure learning)
# ---------------------------------------------------------------------------
#
# Think of three axes people mix-and-match:
#
#   1. Independent vs centralized critics / training
#   2. Cooperative vs competitive vs mixed rewards
#   3. Tabular vs deep function approximation
#
# Common named paradigms:
#
#   Independent Learning (IL)
#     Each agent pretends others are part of the environment.
#     Train Q_i or π_i with ordinary single-agent algorithms (IQL, IPPO, …).
#     Simple and scalable; ignores non-stationarity formally, often still works
#     as a strong baseline.
#
#   Centralized Training, Decentralized Execution (CTDE)
#     During TRAINING, a critic (or planner) may see everyone's info.
#     At EXECUTION, each agent acts from its own observation only.
#     Examples: MADDPG, QMIX, MAPPO. Best of both worlds for many teams.
#
#   Fully Centralized
#     One brain picks the joint action. Optimal in tiny domains; does not scale.
#     Our "Centralized Q-Learning" is this — a teaching upper bound.
#
#   Value factorization (cooperative deep MARL)
#     Learn individual Q_i that sum / mix into a joint Q (VDN, QMIX).
#     Not implemented today; know the name for later reading.
#
#
# ---------------------------------------------------------------------------
# 4) WHAT WE IMPLEMENT (cooperative 2-agent Gridworld)
# ---------------------------------------------------------------------------
#
# Environment sketch (details in later cells):
#   Two agents on a small grid, shared or team reward for reaching goals /
#   coordinating. Fully observed states so tabular methods are honest.
#
#   ┌────────────────────────────────────────────────────────────┐
#   │  IQL   Independent Q-Learning (tabular)                    │
#   │        Each agent keeps its own Q_i(s, a_i).               │
#   │        Update like ordinary Q-learning, treating the other │
#   │        agent as "noise" in the environment.                │
#   │        = simple baseline; can be unstable but often OK.    │
#   ├────────────────────────────────────────────────────────────┤
#   │  Centralized Q  Joint tabular Q(s, a_1, a_2)               │
#   │        One table over joint actions.                       │
#   │        Near-optimal reference when |A|² · |S| still small. │
#   │        = upper bound / "what coordination can achieve".    │
#   ├────────────────────────────────────────────────────────────┤
#   │  IPPO  Independent PPO (neural)                            │
#   │        Each agent runs its own PPO actor–critic (Day 25).  │
#   │        Scales beyond tiny grids; modern default baseline.  │
#   └────────────────────────────────────────────────────────────┘
#
# How to read results later:
#   Centralized Q  ≳  IPPO  ≳  IQL     (typical on a tiny cooperative grid)
#   If IQL ≈ centralized, the task barely needs coordination.
#   If centralized >> IQL, coordination matters — IPPO should close some gap.
#
#
# ---------------------------------------------------------------------------
# BIG PICTURE
# ---------------------------------------------------------------------------
#
#   MDP (1 agent)  →  Markov Game (N agents)
#         │
#         ├─ Independent learners (IQL, IPPO)     ← scalable baselines
#         ├─ Centralized joint Q                  ← tiny-domain upper bound
#         └─ CTDE / value factorization           ← production cooperative MARL
#
# Next cells: build the Gridworld, then IQL → Centralized Q → IPPO.


In [ ]:
# Cooperative Two-Agent Gridworld
# ===============================
#
# TOPIC — the Markov Game we will train on
# ----------------------------------------
# Two agents share a size×size grid. Each picks a move every step.
# They succeed ONLY if BOTH sit on their own goals at the same time:
#
#   Agent 0: start top-left  (0,0)           → goal bottom-right (n-1, n-1)
#   Agent 1: start top-right (0, n-1)        → goal bottom-left  (n-1, 0)
#
#   Reward (team / cooperative):
#     +10  when both on goals together  → episode ends
#     −0.1 every other step             → finish fast
#     timeout at max_steps              → done with no bonus
#
# Why this env for MARL baselines?
#   - Fully observed joint state s = (pos0, pos1)  → tabular Q is honest
#   - Shared reward R0 = R1                        → pure cooperation
#   - Needs a bit of coordination (both must arrive; dawdling costs −0.1)
#   - Small discrete actions {up, down, left, right} each
#
# Formal map (Day 26 concepts):
#   S = all pairs of cell positions
#   A_i = {0,1,2,3}  (indices into self.actions)
#   joint action a = (a0, a1)
#   P = deterministic moves (walls = stay put)
#   R_i = team reward above
#
#
# CODE APPROACH
# -------------
# Setup cell: seeds + torch/numpy imports used by IQL / Centralized Q / IPPO later.
# CoopGridWorld:
#   reset() → joint state tuple
#   step([a0, a1]) → (next_state, [r, r], done)   # same r for both agents


import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from collections import defaultdict
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class CoopGridWorld:
    """Fully cooperative 2-agent grid: both must reach their goals together."""

    def __init__(self, size=5, max_steps=50):
        self.size = size
        self.max_steps = max_steps
        self.actions = ["up", "down", "left", "right"]
        self.action_effects = {
            "up": (-1, 0),
            "down": (1, 0),
            "left": (0, -1),
            "right": (0, 1),
        }
        # Opposite corners: forces agents to cross the board
        self.start = [(0, 0), (0, self.size - 1)]
        self.goals = [(self.size - 1, self.size - 1), (self.size - 1, 0)]
        self.gamma = 0.95
        self.step_reward = -0.1      # cost of time
        self.success_reward = 10.0  # team bonus when both on goals
        self.reset()

    def reset(self):
        self.positions = list(self.start)  # mutable copy of start cells
        self.t = 0
        self.done = False
        return tuple(self.positions)  # joint state s = (pos0, pos1)

    def _move(self, pos, action):
        # Attempt one step; if it would leave the board, stay (bounce off wall)
        di, dj = self.action_effects[action]
        ni, nj = pos[0] + di, pos[1] + dj
        if 0 <= ni < self.size and 0 <= nj < self.size:
            return (ni, nj)
        return pos

    def step(self, actions):
        # actions: [a0, a1] with each a_i an index into self.actions
        assert not self.done
        self.positions[0] = self._move(self.positions[0], self.actions[actions[0]])
        self.positions[1] = self._move(self.positions[1], self.actions[actions[1]])
        self.t += 1

        both_on_goal = (
            self.positions[0] == self.goals[0]
            and self.positions[1] == self.goals[1]
        )
        if both_on_goal:
            reward = self.success_reward
            self.done = True
        else:
            reward = self.step_reward
            if self.t >= self.max_steps:
                self.done = True  # timeout failure (no +10)

        # Same reward for both agents → cooperative Markov game
        return tuple(self.positions), [reward, reward], self.done

# Quick sanity check:
env = CoopGridWorld()
s = env.reset()
print("Initial joint state:", s)
s, r, d = env.step([1, 1])  # both move down
print("After both move down:", s, r, d)

In [ ]:
# Independent Q-Learning (IQL)
# ============================
# TOPIC — each agent learns as if it were alone
# ---------------------------------------------
# IQL = ordinary Q-learning, run once per agent, on the SHARED joint state s.
#
#   Agent i stores Q_i(s, a_i)   ← only ITS action, not the teammate's
#   Update (same as single-agent Q-learning):
#     Q_i(s, a_i) ← Q_i + α [ r + γ max_{a'} Q_i(s', a') − Q_i(s, a_i) ]
#
# The other agent is treated as "part of the environment." That environment
# is NON-STATIONARY (teammate is also learning) — IQL ignores that, which is
# why it is a simple baseline, not a perfect MARL solution.
#
# Exploration: ε-greedy per agent, ε decays over episodes.
# Success metric: fraction of episodes where both finish on their goals.


def iql_train(
    env,
    num_episodes=4000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.2,
    epsilon_decay=0.999,
    min_epsilon=0.02,
    print_every=500,
):
    # One Q-table per agent: key = joint state s, value = array of 4 action values
    Q = [defaultdict(lambda: np.zeros(4)), defaultdict(lambda: np.zeros(4))]
    success_rates = []

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        steps = 0
        while not done:
            # --- Act: each agent ε-greedy on its own Q_i ---
            acts = []
            for i in range(2):
                if random.random() < epsilon:
                    acts.append(random.randrange(4))  # explore
                else:
                    acts.append(int(np.argmax(Q[i][s])))  # exploit

            s_next, rewards, done = env.step(acts)
            r = rewards[0]  # team reward (same for both)

            # --- Learn: independent TD updates (teammate action not in the table) ---
            for i in range(2):
                best_next = 0.0 if done else np.max(Q[i][s_next])
                td_target = r + gamma * best_next
                Q[i][s][acts[i]] += alpha * (td_target - Q[i][s][acts[i]])

            s = s_next
            steps += 1

        # Decay exploration over training
        epsilon = max(min_epsilon, epsilon * epsilon_decay)

        # Did this episode succeed (both on goals when done)?
        success = (
            env.positions[0] == env.goals[0]
            and env.positions[1] == env.goals[1]
        )
        success_rates.append(1 if success else 0)

        if (ep + 1) % print_every == 0:
            rate = np.mean(success_rates[-print_every:])
            print(
                f"IQL Ep {ep+1} | Success rate (last {print_every}): {rate:.2f} "
                f"| eps={epsilon:.3f}"
            )

    return Q, success_rates


In [ ]:
# Centralized Q-Learning (Joint Action)
# =====================================
#
# CONCEPT — fully centralized MARL (tiny-domain upper bound)
# ----------------------------------------------------------
# IQL (previous cell): each agent learns Q_i(s, a_i) and ignores the teammate's
# choice inside its table. Coordination must emerge accidentally.
#
# Centralized Q: treat the TEAM as ONE super-agent whose action is the pair
# (a0, a1). Learn a single table:
#
#   Q(s, a0, a1)   ≈   Q(s, ja)   with ja ∈ {0,…,15}
#
# Encoding (4 actions each → 4² = 16 joint actions):
#   ja = 4*a0 + a1
#   a0 = ja // 4 ,  a1 = ja % 4
#
# Why this matters:
#   + Can represent ANY coordinated strategy (who waits, who goes first, …)
#   + Same math as ordinary Q-learning → easy teaching reference
#   − Joint action space grows as |A|^N  → only OK for small discrete teams
#
# In Day 26 paradigms this is "fully centralized control." Use it as an
# UPPER BOUND: if IQL << Centralized Q, the task needs real coordination.
#
#
# CODE APPROACH
# -------------
# One defaultdict Q[s] → length-16 array
# Loop: ε-greedy over ja → decode (a0,a1) → env.step → TD update on ja
# Track success rate like IQL for fair comparison plots later.


def centralized_q_train(
    env,
    num_episodes=2000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.2,
    epsilon_decay=0.999,
    min_epsilon=0.02,
    print_every=250,
):
    # Joint Q-table: 16 values = every (a0, a1) pair
    Q = defaultdict(lambda: np.zeros(16))
    success_rates = []

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        while not done:
            # Explore / exploit in JOINT action space (not per-agent)
            if random.random() < epsilon:
                ja = random.randrange(16)
            else:
                ja = int(np.argmax(Q[s]))

            # Unpack joint index into the two agents' moves
            a0, a1 = ja // 4, ja % 4
            s_next, rewards, done = env.step([a0, a1])
            r = rewards[0]  # shared team reward

            # Classic Q-learning: bootstrap with max over next joint actions
            best_next = 0.0 if done else np.max(Q[s_next])
            td_target = r + gamma * best_next
            Q[s][ja] += alpha * (td_target - Q[s][ja])
            s = s_next

        epsilon = max(min_epsilon, epsilon * epsilon_decay)

        # Success = both agents on their goals when the episode ended
        success = (
            env.positions[0] == env.goals[0]
            and env.positions[1] == env.goals[1]
        )
        success_rates.append(1 if success else 0)

        if (ep + 1) % print_every == 0:
            rate = np.mean(success_rates[-print_every:])
            print(f"CQL Ep {ep+1} | Success rate: {rate:.2f} | eps={epsilon:.3f}")

    return Q, success_rates


In [ ]:
# Independent PPO (IPPO)
# ======================
#
# CONCEPT — scalable neural baseline for MARL
# -------------------------------------------
# IPPO = each agent runs its OWN PPO actor–critic (Day 25), independently.
#
#   Agent i:  π_i(a_i | s)  and  V_i(s)
#   Same joint state s for both (fully observed Gridworld)
#   Shared team reward r, but SEPARATE networks / optimizers
#
# Like IQL, teammates are treated as part of the environment (non-stationary).
# Unlike IQL, policies are neural nets → scales beyond tiny tabular grids.
#
# vs other baselines in this notebook:
#   IQL            — tabular independent Q (simple, limited)
#   Centralized Q  — joint action table (upper bound, tiny only)
#   IPPO           — deep independent policy gradient + clip + GAE
#
# This is "independent learning" in the Day 26 paradigms — NOT CTDE
# (no centralized critic that sees both agents' internals).
#
#
# CODE MAP
# --------
# ActorCritic     — shared trunk → actor logits + critic V  (per agent)
# state_to_vec    — one-hot both positions → fixed-size input vector
# compute_gae     — advantages for PPO (same idea as Day 25)
# ippo_train      — collect joint rollouts → PPO update each agent


class ActorCritic(nn.Module):
    """One agent's policy π(a|s) + value V(s). Instantiated twice for IPPO."""

    def __init__(self, state_dim, n_actions, hidden_dim=64):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.actor = nn.Linear(hidden_dim, n_actions)
        self.critic = nn.Linear(hidden_dim, 1)
        # Orthogonal init (PPO-stable defaults)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)  # start near-uniform π
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h).squeeze(-1)


def state_to_vec(joint_state, size=5):
    """Encode s = (pos0, pos1) as concatenated one-hots → length 2*size*size."""
    vec = np.zeros(2 * size * size, dtype=np.float32)
    for i, (r, c) in enumerate(joint_state):
        vec[i * size * size + r * size + c] = 1.0
    return vec


def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    """Backward GAE: Â_t = δ_t + γλ(1−done) Â_{t+1} ; returns = Â + V."""
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last = 0.0
    for t in reversed(range(T)):
        nv = next_value if t == T - 1 else values[t + 1]
        nt = 1.0 - dones[t]
        delta = rewards[t] + gamma * nv * nt - values[t]
        last = delta + gamma * lam * nt * last
        adv[t] = last
    returns = adv + values
    return adv, returns


def ippo_train(
    env,
    num_updates=200,
    rollout_len=512,
    gamma=0.95,
    lam=0.95,
    clip_eps=0.2,
    value_coef=0.5,
    entropy_coef=0.01,
    ppo_epochs=6,
    lr=3e-4,
    print_every=20,
):
    state_dim = 2 * env.size * env.size
    # Independent learners: separate nets + Adam for each agent
    policies = [ActorCritic(state_dim, 4).to(device) for _ in range(2)]
    optimizers = [optim.Adam(p.parameters(), lr=lr) for p in policies]

    success_rates = []

    for update in range(num_updates):
        # --- 1) Collect a joint rollout (both agents act every step) ---
        bufs = [[] for _ in range(2)]  # per-agent transition lists
        s = env.reset()
        ep_successes = []
        steps = 0
        while steps < rollout_len:
            sv = state_to_vec(s, env.size)
            sv_t = torch.FloatTensor(sv).unsqueeze(0).to(device)
            acts, logps, vals = [], [], []
            for i in range(2):
                with torch.no_grad():
                    logits, v = policies[i](sv_t)
                    dist = Categorical(logits=logits)
                    a = dist.sample()
                    acts.append(a.item())
                    logps.append(dist.log_prob(a).item())  # π_old for ratio
                    vals.append(v.item())
            s_next, rewards, done = env.step(acts)
            r = rewards[0]  # team reward copied into both buffers
            for i in range(2):
                bufs[i].append((sv, acts[i], logps[i], r, vals[i], float(done)))
            s = s_next
            steps += 1
            if done:
                success = (
                    env.positions[0] == env.goals[0]
                    and env.positions[1] == env.goals[1]
                )
                ep_successes.append(1 if success else 0)
                s = env.reset()

        # --- 2) PPO update EACH agent on its own buffer (independent) ---
        for i in range(2):
            states = torch.tensor(
                np.stack([b[0] for b in bufs[i]]), dtype=torch.float32, device=device
            )
            actions = torch.tensor(
                [b[1] for b in bufs[i]], dtype=torch.long, device=device
            )
            old_logp = torch.tensor(
                [b[2] for b in bufs[i]], dtype=torch.float32, device=device
            )
            rewards = [b[3] for b in bufs[i]]
            values = [b[4] for b in bufs[i]]
            dones = [b[5] for b in bufs[i]]

            # Bootstrap V(s_T) for GAE at end of fragment
            with torch.no_grad():
                _, nv = policies[i](
                    torch.tensor(
                        state_to_vec(s, env.size), dtype=torch.float32, device=device
                    ).unsqueeze(0)
                )
                next_v = nv.item()

            adv, ret = compute_gae(rewards, values, dones, next_v, gamma, lam)
            adv = torch.tensor(adv, dtype=torch.float32, device=device)
            ret = torch.tensor(ret, dtype=torch.float32, device=device)
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)

            # Multi-epoch clipped PPO (same loss as Day 25)
            for _ in range(ppo_epochs):
                logits, vs = policies[i](states)
                dist = Categorical(logits=logits)
                new_logp = dist.log_prob(actions)
                ratio = torch.exp(new_logp - old_logp)
                s1 = ratio * adv
                s2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
                ploss = -torch.min(s1, s2).mean()
                vloss = F.mse_loss(vs, ret)
                ent = dist.entropy().mean()
                loss = ploss + value_coef * vloss - entropy_coef * ent
                optimizers[i].zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policies[i].parameters(), 0.5)
                optimizers[i].step()

        if ep_successes:
            success_rates.extend(ep_successes)
        if (update + 1) % print_every == 0:
            recent = success_rates[-200:]
            print(
                f"IPPO Update {update+1} | "
                f"Success rate (last {len(recent)}): {np.mean(recent):.2f}"
            )

    return policies, success_rates

# ---------------------------------------------------------------------------
# Train and Compare — run all three MARL baselines on fresh Gridworlds
# ---------------------------------------------------------------------------
# Fresh env per method so episode counters / positions don't leak across runs.
# Each returns a success_rates list (1/0 per episode) for later plotting.
#
# Expected rough ordering on this cooperative task:
#   Centralized Q  ≳  IPPO  ≳  IQL
# (joint Q can coordinate; IPPO scales; IQL is the simplest independent baseline)
#
# Episode/update counts are demo-sized — raise them for smoother curves.

# IQL: tabular independent Q-learning (~3000 episodes)
env = CoopGridWorld()
Q_iql, sr_iql = iql_train(env, num_episodes=3000)
print()

# Centralized Q: joint-action table (fewer episodes often enough — denser updates)
env = CoopGridWorld()
Q_cql, sr_cql = centralized_q_train(env, num_episodes=1500)
print()

# IPPO: two neural PPO agents, 150 updates × 512 rollout steps each
env = CoopGridWorld()
policies_ippo, sr_ippo = ippo_train(env, num_updates=150, rollout_len=512)


# Plot smoothed success rates side‑by‑side:
def smooth(x, w=100):
    if len(x) < w: return x
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 5))
plt.plot(smooth(sr_iql), label='IQL')
plt.plot(smooth(sr_cql), label='Centralized Q')
plt.plot(smooth(sr_ippo), label='IPPO')
plt.xlabel('Episode / update (rolling window)')
plt.ylabel('Success rate')
plt.title('Cooperative Two-Agent Gridworld — MARL Baselines')
plt.legend()
plt.ylim(0, 1.05)
plt.show()

In [ ]:
# =============================================================================
# HANDS-ON MINI-PROJECT — MARL Baselines
# =============================================================================
# Builds on CoopGridWorld + IQL / Centralized Q / IPPO from earlier cells.
#
# 3.1 Coordinated start     — both agents spawn at (0,0); needs coordination
# 3.2 Communication channel — discrete messages (tiny CommNet-style ablation)
# 3.3 Competitive variant   — race: first to own goal wins
# 3.4 PettingZoo cross-check — optional simple_spread note
# 3.5 Portfolio wrap-up     — curves, checkpoints, CTDE / non-stationarity discussion
#
# Artifacts → week4/artifacts_day26/


In [ ]:
# -----------------------------------------------------------------------------
# 3.1 Coordinated vs Independent Start
# -----------------------------------------------------------------------------
# CONCEPT
# -------
# Default env: agents start on opposite corners → paths barely interact.
# Coordinated start: BOTH begin at (0,0), goals still (4,4) and (4,0).
#
# Now they occupy the same cell at t=0 and must SEPARATE toward different goals.
# Credit assignment / coordination matter more:
#   - IQL may struggle (each Q_i cannot condition on teammate's action)
#   - Centralized Q can still plan joint moves → should stay stronger
#
# We subclass CoopGridWorld and only change start positions.


class CoordinatedStartGrid(CoopGridWorld):
    """Same goals/rewards; both agents spawn at top-left (0,0)."""

    def __init__(self, size=5, max_steps=50):
        super().__init__(size=size, max_steps=max_steps)
        self.start = [(0, 0), (0, 0)]  # stacked start
        self.reset()


print("=== 3.1 Default starts (reminder from main train) ===")
print("IQL last-200 success:", f"{np.mean(sr_iql[-200:]):.2f}" if len(sr_iql) >= 200 else np.mean(sr_iql))
print("CQL last-200 success:", f"{np.mean(sr_cql[-200:]):.2f}" if len(sr_cql) >= 200 else np.mean(sr_cql))

print("\n=== 3.1 Coordinated start (0,0)/(0,0) → goals (4,4)/(4,0) ===")
env_cs = CoordinatedStartGrid()
print("start=", env_cs.start, "goals=", env_cs.goals)

Q_iql_cs, sr_iql_cs = iql_train(env_cs, num_episodes=3000, print_every=500)
env_cs = CoordinatedStartGrid()
Q_cql_cs, sr_cql_cs = centralized_q_train(env_cs, num_episodes=1500, print_every=250)

def _tail(x, n=200):
    return float(np.mean(x[-n:])) if len(x) else float("nan")

print("\n3.1 summary (last-200 success rate)")
print(f"  IQL  coordinated-start: {_tail(sr_iql_cs):.2f}  | default IQL: {_tail(sr_iql):.2f}")
print(f"  CQL  coordinated-start: {_tail(sr_cql_cs):.2f}  | default CQL: {_tail(sr_cql):.2f}")
print("If CQL >> IQL here, coordination / joint action was the bottleneck.")

plt.figure(figsize=(10, 4))
w = 100
plt.plot(np.convolve(sr_iql_cs, np.ones(w)/w, mode="valid"), label="IQL (coord start)")
plt.plot(np.convolve(sr_cql_cs, np.ones(w)/w, mode="valid"), label="CQL (coord start)")
plt.xlabel("Episode"); plt.ylabel("Success rate")
plt.title("3.1 Coordinated start — IQL vs Centralized Q")
plt.legend(); plt.ylim(0, 1.05); plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# 3.2 Communication Ablation (CommNet-style discrete messages)
# -----------------------------------------------------------------------------
# CONCEPT
# -------
# Each agent also emits m_i ∈ {0,1,2,3} every step. The teammate observes that
# message as part of its Q-key:
#   key_i = (joint_state, teammate_message)
# Action space per agent becomes (move × message) = 4×4 = 16 choices.
#
# This is a minimal "learned communication" ablation (CommNet spirit).
# We test on the COORDINATED-START grid where coordination is harder —
# messages should help IQL if agents learn to signal intent (e.g. "I go right").


class CommGridWorld(CoordinatedStartGrid):
    """Coordinated-start grid that stores the latest message from each agent."""

    def __init__(self, size=5, max_steps=50, n_messages=4):
        self.n_messages = n_messages
        super().__init__(size=size, max_steps=max_steps)
        self.last_msgs = [0, 0]

    def reset(self):
        s = super().reset()
        self.last_msgs = [0, 0]
        return s

    def step(self, actions_and_msgs):
        # actions_and_msgs: [(a0, m0), (a1, m1)]
        self.last_msgs = [int(actions_and_msgs[0][1]), int(actions_and_msgs[1][1])]
        return super().step([actions_and_msgs[0][0], actions_and_msgs[1][0]])


def iql_train_comm(
    env,
    num_episodes=3000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.2,
    epsilon_decay=0.999,
    min_epsilon=0.02,
    print_every=500,
):
    n_act, n_msg = 4, env.n_messages
    n_joint = n_act * n_msg  # pick move AND message together
    Q = [defaultdict(lambda: np.zeros(n_joint)) for _ in range(2)]
    success_rates = []

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        while not done:
            seen = list(env.last_msgs)  # teammate signal from previous step
            keys, jas, am = [], [], []
            for i in range(2):
                key = (s, seen[1 - i])
                keys.append(key)
                ja = (
                    random.randrange(n_joint)
                    if random.random() < epsilon
                    else int(np.argmax(Q[i][key]))
                )
                jas.append(ja)
                am.append((ja // n_msg, ja % n_msg))  # (action, message)

            s_next, rewards, done = env.step(am)
            r = rewards[0]
            new_seen = list(env.last_msgs)
            for i in range(2):
                next_key = (s_next, new_seen[1 - i])
                best_next = 0.0 if done else np.max(Q[i][next_key])
                Q[i][keys[i]][jas[i]] += alpha * (
                    (r + gamma * best_next) - Q[i][keys[i]][jas[i]]
                )
            s = s_next

        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        success = (
            env.positions[0] == env.goals[0]
            and env.positions[1] == env.goals[1]
        )
        success_rates.append(1 if success else 0)
        if (ep + 1) % print_every == 0:
            print(
                f"IQL+comm Ep {ep+1} | "
                f"success={np.mean(success_rates[-print_every:]):.2f} | eps={epsilon:.3f}"
            )
    return Q, success_rates


print("=== 3.2 IQL + communication on coordinated-start ===")
env_comm = CommGridWorld()
Q_comm, sr_comm = iql_train_comm(env_comm, num_episodes=3000)

print("\n3.2 summary (last-200 success on coordinated start)")
print(f"  IQL plain:  {_tail(sr_iql_cs):.2f}")
print(f"  IQL + comm: {_tail(sr_comm):.2f}")

plt.figure(figsize=(10, 4))
w = 100
plt.plot(np.convolve(sr_iql_cs, np.ones(w) / w, mode="valid"), label="IQL (no comm)")
plt.plot(np.convolve(sr_comm, np.ones(w) / w, mode="valid"), label="IQL + messages")
plt.title("3.2 Communication ablation (coordinated start)")
plt.xlabel("Episode")
plt.ylabel("Success rate")
plt.legend()
plt.ylim(0, 1.05)
plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# 3.3 Competitive Variant — race to your own goal
# -----------------------------------------------------------------------------
# CONCEPT
# -------
# Switch from shared team reward to a RACE:
#   +1 to whoever reaches THEIR goal first (and episode ends)
#   0 otherwise / step cost optional
#
# This is a mixed / competitive Markov game: agents may block, race, or
# "accidently" cooperate. IQL / IPPO still run, but success is no longer
# "both on goals" — we track who wins.


class CompetitiveGrid(CoopGridWorld):
    """First agent to reach its own goal gets +1; other gets 0; episode ends."""

    def step(self, actions):
        assert not self.done
        self.positions[0] = self._move(self.positions[0], self.actions[actions[0]])
        self.positions[1] = self._move(self.positions[1], self.actions[actions[1]])
        self.t += 1

        g0 = self.positions[0] == self.goals[0]
        g1 = self.positions[1] == self.goals[1]

        if g0 and g1:
            # simultaneous → both score (rare tie)
            rewards = [1.0, 1.0]
            self.done = True
        elif g0:
            rewards = [1.0, 0.0]
            self.done = True
        elif g1:
            rewards = [0.0, 1.0]
            self.done = True
        else:
            rewards = [-0.01, -0.01]  # small time pressure
            if self.t >= self.max_steps:
                self.done = True
        return tuple(self.positions), rewards, self.done


def iql_train_competitive(
    env,
    num_episodes=3000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.2,
    epsilon_decay=0.999,
    min_epsilon=0.02,
    print_every=500,
):
    """IQL with per-agent rewards (no longer assuming rewards[0] == rewards[1])."""
    Q = [defaultdict(lambda: np.zeros(4)), defaultdict(lambda: np.zeros(4))]
    wins = [[], []]  # who won each episode

    for ep in range(num_episodes):
        s = env.reset()
        done = False
        last_rewards = [0.0, 0.0]
        while not done:
            acts = [
                random.randrange(4)
                if random.random() < epsilon
                else int(np.argmax(Q[i][s]))
                for i in range(2)
            ]
            s_next, rewards, done = env.step(acts)
            last_rewards = rewards
            for i in range(2):
                best_next = 0.0 if done else np.max(Q[i][s_next])
                td = rewards[i] + gamma * best_next
                Q[i][s][acts[i]] += alpha * (td - Q[i][s][acts[i]])
            s = s_next

        epsilon = max(min_epsilon, epsilon * epsilon_decay)
        # Record winner from terminal rewards
        if last_rewards[0] > 0 and last_rewards[1] > 0:
            wins[0].append(1); wins[1].append(1)  # tie
        elif last_rewards[0] > 0:
            wins[0].append(1); wins[1].append(0)
        elif last_rewards[1] > 0:
            wins[0].append(0); wins[1].append(1)
        else:
            wins[0].append(0); wins[1].append(0)  # timeout

        if (ep + 1) % print_every == 0:
            print(
                f"Comp-IQL Ep {ep+1} | "
                f"P0 win-rate={np.mean(wins[0][-print_every:]):.2f} | "
                f"P1 win-rate={np.mean(wins[1][-print_every:]):.2f} | "
                f"eps={epsilon:.3f}"
            )
    return Q, wins


print("=== 3.3 Competitive race ===")
env_comp = CompetitiveGrid()
Q_comp, wins_comp = iql_train_competitive(env_comp, num_episodes=3000)

plt.figure(figsize=(10, 4))
w = 100
plt.plot(np.convolve(wins_comp[0], np.ones(w)/w, mode="valid"), label="Agent 0 win rate")
plt.plot(np.convolve(wins_comp[1], np.ones(w)/w, mode="valid"), label="Agent 1 win rate")
plt.title("3.3 Competitive Grid — IQL win rates (equilibrium / race)")
plt.xlabel("Episode"); plt.ylabel("Win rate"); plt.legend(); plt.ylim(0, 1.05)
plt.show()
print(
    "Late win rates — "
    f"P0={np.mean(wins_comp[0][-200:]):.2f}, P1={np.mean(wins_comp[1][-200:]):.2f} "
    "(similar rates ≈ rough equilibrium of racing policies)"
)


In [ ]:
# -----------------------------------------------------------------------------
# 3.4 PettingZoo Cross-Check (optional)
# -----------------------------------------------------------------------------
# CONCEPT
# -------
# PettingZoo is the multi-agent cousin of Gymnasium. `simple_spread_v3` is a
# classic cooperative particle benchmark (cover landmarks without colliding).
# Replicating IPPO there is a strong portfolio stretch goal.
#
# We try to import; if missing, print install instructions (no hard failure).


try:
    from pettingzoo.mpe import simple_spread_v3

    PZ_OK = True
    env_pz = simple_spread_v3.parallel_env(N=2, max_cycles=25, continuous_actions=False)
    obs, infos = env_pz.reset(seed=42)
    print("PettingZoo simple_spread_v3 OK.")
    print("Agents:", env_pz.agents)
    print("Obs shapes:", {a: np.shape(o) for a, o in obs.items()})
    print(
        "Next step for portfolio: wrap parallel_env → IPPO rollout "
        "(same clipped loss as Day 25 / this notebook's ippo_train)."
    )
    env_pz.close()
except ImportError:
    PZ_OK = False
    print("PettingZoo not installed — optional for 3.4.")
    print("  pip install pettingzoo")
    print("Then: from pettingzoo.mpe import simple_spread_v3")


In [ ]:
# -----------------------------------------------------------------------------
# 3.5 Portfolio Artifact — discussion, ASCII diagram, checkpoints
# -----------------------------------------------------------------------------
#
# ASCII: cooperative episode sketch (default opposite-corner starts)
#
#   t=0                         mid                          success
#   A0 .  .  .  A1              .  .  A0 .  .               .  .  .  .
#   .  .  .  .  .               .  .  .  .  .               .  .  .  .
#   .  .  .  .  .      →        .  A1 .  .  .       →       .  .  .  .
#   .  .  .  .  .               .  .  .  .  .               G1 .  .  G0
#   G1 .  .  .  G0              G1 .  .  .  G0              (both on goals)
#
# Coordinated start (3.1): both A0 and A1 stack on top-left at t=0.
#
#
# DISCUSSION
# ----------
# Non-stationarity:
#   In IQL/IPPO each agent updates while the other also changes → the
#   "environment" from i's view keeps shifting. Centralized Q avoids this
#   by controlling the joint action directly (but does not scale).
#
# Credit assignment:
#   Shared +10 only when BOTH succeed. Independent learners may not know
#   whose move mattered. Comm (3.2) and joint Q (CQL) are two ways to help.
#
# CTDE (Centralized Training, Decentralized Execution):
#   Not implemented here, but the production pattern is: train with a critic
#   that sees everyone (like MAPPO / QMIX), execute with local policies only.
#   IPPO ≈ decentralized train+exec; CQL ≈ centralized train+exec;
#   CTDE sits in between.
#
# Competitive (3.3):
#   Win-rate curves near 50/50 suggest agents learned racing policies that
#   roughly equilibrate — a taste of game-theoretic MARL.


from pathlib import Path

ARTIFACT_DIR = Path("/Users/girish11/aifromscratch_code/week4/artifacts_day26")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Save IPPO policies from the main cooperative training run
if "policies_ippo" in dir() and policies_ippo is not None:
    torch.save(policies_ippo[0].state_dict(), ARTIFACT_DIR / "ippo_agent0.pt")
    torch.save(policies_ippo[1].state_dict(), ARTIFACT_DIR / "ippo_agent1.pt")
    print("Saved:", ARTIFACT_DIR / "ippo_agent0.pt")
    print("Saved:", ARTIFACT_DIR / "ippo_agent1.pt")
else:
    print("policies_ippo missing — re-run the Train-and-Compare cell first.")

# Save comparison figure for the main baselines
fig = plt.figure(figsize=(10, 5))
w = 100
plt.plot(np.convolve(sr_iql, np.ones(w)/w, mode="valid"), label="IQL")
plt.plot(np.convolve(sr_cql, np.ones(w)/w, mode="valid"), label="Centralized Q")
plt.plot(np.convolve(sr_ippo, np.ones(w)/w, mode="valid"), label="IPPO")
plt.xlabel("Episode / update (rolling)")
plt.ylabel("Success rate")
plt.title("Day 26 — Cooperative Gridworld MARL Baselines")
plt.legend(); plt.ylim(0, 1.05)
fig.savefig(ARTIFACT_DIR / "fig_baselines.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", ARTIFACT_DIR / "fig_baselines.png")

# Write a short results stub (filled more by the offline runner if present)
summary = f"""# Day 26 Results

## Main cooperative baselines (default opposite-corner starts)
- IQL last-200 success:  {_tail(sr_iql):.2f}
- CQL last-200 success:  {_tail(sr_cql):.2f}
- IPPO last-200 success: {_tail(sr_ippo):.2f}

## 3.1 Coordinated start
- IQL: {_tail(sr_iql_cs):.2f}
- CQL: {_tail(sr_cql_cs):.2f}

## 3.2 Communication (coord start)
- IQL plain: {_tail(sr_iql_cs):.2f}
- IQL+comm:  {_tail(sr_comm):.2f}

## 3.3 Competitive late win rates
- P0: {np.mean(wins_comp[0][-200:]):.2f}
- P1: {np.mean(wins_comp[1][-200:]):.2f}

Checkpoints: ippo_agent0.pt, ippo_agent1.pt
"""
(ARTIFACT_DIR / "RESULTS.md").write_text(summary)
print(summary)
